# Module 2: Sampling -- From Continuous Body to Discrete Computer

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup

Import our standard libraries for this module.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Ready!')

**Exercise 1:** Try it yourself!

## Step 2: From Continuous to Discrete

**Clinical scenario:** A technician is setting up an EEG for a patient with suspected temporal lobe epilepsy. The attending neurologist asks: "What sampling rate are you using?" The tech replies: "256 Hz." Why 256? Why not 100? Why not 10,000? And what happens if you get it wrong?

Your body produces **continuous** signals -- voltage that changes every infinitesimal instant. Computers only understand **discrete numbers** -- a list of measurements at specific times.

**Sampling** means measuring the voltage at regular intervals. The rate at which you "check" the signal is called the **sampling rate** ($f_s$), measured in Hz (samples per second).

**Exercise 2:** Try it yourself!

## Step 3: Visualizing Sampling

Let's see what happens when we sample a continuous signal at different rates. The continuous signal (blue) is the "truth." The red dots are the discrete samples the computer captures.

In [ ]:
def plot_sampling(signal_freq, sampling_rate, duration=1.0):
    """Show continuous signal with discrete samples overlaid."""
    # 'Continuous' signal (high resolution)
    t_cont = np.arange(0, duration, 1/2000)
    y_cont = np.sin(2 * np.pi * signal_freq * t_cont)
    
    # Sampled points
    t_samp = np.arange(0, duration, 1/sampling_rate)
    y_samp = np.sin(2 * np.pi * signal_freq * t_samp)
    
    # Determine status
    aliased = sampling_rate < 2 * signal_freq
    status = 'ALIASED!' if aliased else 'OK'
    status_color = '#c0392b' if aliased else '#27ae60'
    
    plt.figure(figsize=(10, 3.5))
    plt.plot(t_cont, y_cont, color='#2980b9', linewidth=2, label='Continuous signal')
    plt.plot(t_samp, y_samp, 'o', color='#e74c3c', markersize=6, label=f'Samples (fs={sampling_rate} Hz)')
    # Connect samples with dashed line (reconstruction)
    plt.plot(t_samp, y_samp, '--', color='#e74c3c', linewidth=1, alpha=0.5, label='Reconstruction')
    
    plt.title(f'{signal_freq} Hz signal sampled at {sampling_rate} Hz -- {status}',
              color=status_color, fontweight='bold')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.ylim(-1.5, 1.5)
    plt.tight_layout()
    plt.show()

# Good sampling: 5 Hz signal at 40 Hz
plot_sampling(signal_freq=5, sampling_rate=40)

# Marginal: 5 Hz signal at 12 Hz
plot_sampling(signal_freq=5, sampling_rate=12)

# Undersampled: 5 Hz signal at 8 Hz
plot_sampling(signal_freq=5, sampling_rate=8)

**Exercise 3:** Try it yourself!

## Step 4: Nyquist Demo: Aliasing a 10 Hz Signal

**The Nyquist-Shannon Sampling Theorem:** To faithfully capture a signal of frequency $f$, you must sample at *at least* $2f$ Hz.

$$f_s \geq 2 \times f_{max}$$

Below that threshold, the signal **aliases** -- it masquerades as a lower frequency. The aliased frequency is: $f_{alias} = |f_{signal} - n \cdot f_s|$ where $n$ is the nearest integer.

Let's sample a fixed 10 Hz signal at three different rates and see what happens.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

f_signal = 10  # Hz
t_cont = np.arange(0, 1.0, 1/2000)  # 'Continuous'
y_cont = np.sin(2 * np.pi * f_signal * t_cont)

rates = [
    (50, '#27ae60', 'fs=50 Hz (good) -- well above 2x10=20 Hz'),
    (12, '#e67e22', 'fs=12 Hz (marginal) -- barely above Nyquist'),
    (8, '#c0392b', 'fs=8 Hz (ALIASED) -- below 2x10=20 Hz'),
]

for ax, (fs, color, title) in zip(axes, rates):
    t_samp = np.arange(0, 1.0, 1/fs)
    y_samp = np.sin(2 * np.pi * f_signal * t_samp)
    
    ax.plot(t_cont, y_cont, color='#2980b9', linewidth=2, alpha=0.5, label='True 10 Hz')
    ax.plot(t_samp, y_samp, 'o-', color=color, markersize=6, linewidth=1.5, label=f'Sampled')
    
    # Show alias if undersampled
    if fs < 2 * f_signal:
        f_alias = abs(f_signal - round(f_signal / fs) * fs)
        if f_alias > 0:
            y_alias = np.sin(2 * np.pi * f_alias * t_cont)
            ax.plot(t_cont, y_alias, '--', color=color, linewidth=2,
                    label=f'Alias: {f_alias:.1f} Hz (phantom!)')
    
    ax.set_title(title, fontsize=11, color=color)
    ax.legend(fontsize=9)
    ax.set_ylim(-1.5, 1.5)

axes[-1].set_xlabel('Time (seconds)')
plt.suptitle('Sampling a 10 Hz Signal at Different Rates', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Exercise 4:** Try it yourself!

## Step 5: Alias Frequency Calculation

The alias frequency can be computed mathematically. When you undersample:

$$f_{alias} = |f_{signal} - n \cdot f_s|$$

where $n = \text{round}(f_{signal} / f_s)$.

This is dangerous because the alias **looks perfectly real**. A 60 Hz power line signal sampled at 50 Hz would appear as 10 Hz -- which could be mistaken for alpha rhythm in an EEG!

In [ ]:
def compute_alias(f_signal, f_sample):
    """Compute the aliased frequency."""
    n = round(f_signal / f_sample)
    f_alias = abs(f_signal - n * f_sample)
    return f_alias

# Examples
examples = [
    (60, 50, '60 Hz noise sampled at 50 Hz'),
    (10, 8, '10 Hz signal sampled at 8 Hz'),
    (100, 75, '100 Hz signal sampled at 75 Hz'),
    (30, 25, '30 Hz signal sampled at 25 Hz'),
]

print(f'{"Signal":>10s}  {"Sample Rate":>12s}  {"Nyquist OK?":>12s}  {"Alias Freq":>12s}')
print('-' * 52)
for f_sig, f_samp, desc in examples:
    f_alias = compute_alias(f_sig, f_samp)
    ok = 'Yes' if f_samp >= 2 * f_sig else 'NO'
    print(f'{f_sig:>8d} Hz  {f_samp:>10d} Hz  {ok:>12s}  {f_alias:>10.1f} Hz')
    print(f'  --> {desc}')

**Exercise 5:** Try it yourself!

## Step 6: Clinical Sampling Rates

Nobody samples at exactly $2f_{max}$ -- that's the bare minimum. Real systems add a generous safety margin:

| Signal | Frequencies of Interest | Min Nyquist | Typical Clinical Rate |
|---|---|---|---|
| EEG | 0.5-100 Hz | 200 Hz | 256-512 Hz |
| ECG (Holter) | 0.5-150 Hz | 300 Hz | 128-500 Hz |
| EMG | 20-500 Hz | 1000 Hz | 1000-5000 Hz |

Why 256 Hz for EEG? Because the highest frequency neurologists typically care about is ~100 Hz. $2 \times 100 = 200$ Hz minimum. 256 gives a comfortable margin and happens to be a power of 2 (which makes FFT computation faster).

In [ ]:
def sampling_calculator(signal_name, f_max):
    """Calculate recommended sampling rates."""
    nyquist = 2 * f_max
    practical = 4 * f_max
    # Nearest power of 2
    power_of_2 = 2 ** int(np.ceil(np.log2(practical)))
    
    print(f'\n{signal_name}')
    print(f'  Max frequency of interest: {f_max} Hz')
    print(f'  Minimum Nyquist rate: 2 x {f_max} = {nyquist} Hz')
    print(f'  Recommended (4x): {practical} Hz')
    print(f'  Nearest power-of-2: {power_of_2} Hz')

sampling_calculator('EEG', 100)
sampling_calculator('ECG (Holter)', 150)
sampling_calculator('EMG', 500)

**Exercise 6:** Try it yourself!

## Step 7: Exercise: Minimum Sampling Rate

Apply the Nyquist theorem to determine the minimum sampling rate.

**Exercise 7:** Try it yourself!

In [ ]:
# YOUR CODE HERE
# Task: What is the minimum sampling rate for a 40 Hz EEG signal?
#
# Calculate it and print your answer:
# f_max = 40  # Hz
# f_nyquist = ???  # What must this be?
# print(f'Minimum sampling rate for {f_max} Hz: {f_nyquist} Hz')
#
# Bonus: what would you actually use in clinical practice? (Hint: 4-5x margin)


## Step 8: Exercise: Aliasing in Practice

Sample a 30 Hz sine wave at 25 Hz and visualize what happens. Calculate the alias frequency mathematically, then verify it visually.

**Exercise 8:** Try it yourself!

In [ ]:
# YOUR CODE HERE
# Task: Sample a 30 Hz sine at 25 Hz. What frequency does the alias appear at?
#
# Steps:
# 1. Compute: f_alias = abs(f_signal - round(f_signal / f_sample) * f_sample)
# 2. Generate the continuous 30 Hz signal (use fs=2000 for 'continuous')
# 3. Sample it at 25 Hz (t_samp = np.arange(0, 1.0, 1/25))
# 4. Generate the alias frequency sine wave
# 5. Plot all three: continuous, samples, and alias
#
# Expected alias: |30 - 1*25| = 5 Hz. Verify this in your plot!
